# Computational Theory for ML — Module 1 Foundations

## Learning Objectives
- Understand the mathematical models of computation that underlie ML algorithms
- Grasp finite automata, Turing machines, and complexity classes (P, NP)
- Connect computational theory to practical ML limitations and design choices
- Recognize when a problem's computational complexity affects feasibility

## Why This Matters
Skillsoft includes 1h 50m of computational theory covering finite automata, Turing machines, context-free grammars, and complexity classes. This is unique to the ML Architect path — engineers rarely study this, but architects must understand computational limits to make informed design choices.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import functools
import operator
sns.set_style(\"whitegrid\")
np.random.seed(42)
%matplotlib inline


## 1. Finite Automata (DFA & NFA)

A **finite automaton** is a simple computational model with a finite set of states, transitions, and accepting states. DFAs (deterministic) and NFAs (non-deterministic) recognize regular languages — the simplest class in the Chomsky hierarchy.


In [ ]:
class DFAEvenOnes:
    \"\"\"DFA accepting binary strings with even number of 1s.

    States: q0 (even count so far), q1 (odd count so far)
    Start: q0 (0 ones is even)
    Accept: q0
    Transitions:
      - From q0, read '0' -> stay in q0
      - From q0, read '1' -> go to q1
      - From q1, read '0' -> stay in q1
      - From q1, read '1' -> go to q0
    \"\"\"

    def __init__(self):
        self.states = {'q0', 'q1'}
        self.start_state = 'q0'
        self.accept_states = {'q0'}
        # Transition function: (state, input) -> next_state
        self.transitions = {
            ('q0', '0'): 'q0',
            ('q0', '1'): 'q1',
            ('q1', '0'): 'q1',
            ('q1', '1'): 'q0',
        }
    
    def accepts(self, string):
        \"\"\"Return True if the DFA accepts the input string.\"\"\"
        state = self.start_state
        for char in string:
            state = self.transitions[(state, char)]
        return state in self.accept_states


### 1.1 DFA: Binary Even Parity Detector

A DFA that accepts binary strings with an even number of 1s.


In [ ]:
# Test the DFA
dfa = DFAEvenOnes()
test_strings = [
    (\"\", \"empty string (0 ones, even)\"),
    (\"0\", \"single zero\"),
    (\"1\", \"single one (odd)\"),
    (\"00\", \"two zeros\"),
    (\"01\", \"zero then one\"),
    (\"10\", \"one then zero\"),
    (\"11\", \"two ones (even)\"),
    (\"101\", \"one-zero-one (even)\"),
    (\"111\", \"three ones (odd)\"),
    (\"01010\", \"alternating (even)\"),
]

print(\"DFA Even-Odd Parity Detector\")
print(\"=\" * 60)
for s, desc in test_strings:
    result = dfa.accepts(s)
    print(f\"  '{s:>8s}' : {desc:40s} -> {'ACCEPT' if result else 'REJECT'}\")


### 1.2 Visualizing the DFA

DOT format for graph visualization.


In [ ]:
def draw_dfa(dfa, filename=\"dfa_even_ones\"):
    \"\"\"Draw the DFA as a simple graph.\"\"\"
    state_colors = {'q0': 'lightgreen', 'q1': 'lightcoral'}

    dot_lines = []
    dot_lines.append(\"digraph DFA {\")
    dot_lines.append(\"  rankdir=LR;\")
    dot_lines.append(\"  size=\"4,4\";\")
    dot_lines.append(\"  node [shape=circle, style=filled];\")

    # Start node arrow
    dot_lines.append(f\"  start [shape=none label=\"\"\"\"];\")


In [ ]:
# Verify: all accepted strings have even number of 1s
print(\"\
Verification: Checking all 4-bit strings\")
all_even = sum(1 for i in range(16) if dfa.accepts(format(i, '04b')))
print(f\"  {all_even}/16 four-bit strings accepted (expected 8 with even parity)\")


## 2. NFA: Non-Deterministic Finite Automaton

An NFA can be in multiple states simultaneously. It accepts if *any* path leads to an accept state. NFAs are equivalent in power to DFAs (every NFA has an equivalent DFA), but often more concise.

### Exercise: Convert NFA to DFA

Given the NFA that accepts strings over {0,1} ending in '01':
- States: {q0, q1, q2}
- Start: q0
- Accept: q2
- Transitions: q0 -0-> q0, q0 -1-> q0, q0 -0-> q1, q1 -1-> q2, q2 -0-> q0, q2 -1-> q0
- Question: What is the minimum DFA equivalent to this NFA?


In [ ]:
# NFA simulator: tracks set of current states
class NFAEndsIn01:
    def __init__(self):
        self.states = [{'q0'}, {'q0', 'q1'}, {'q2'}, set()]
        # State mapping: 0={q0}, 1={q0,q1}, 2={q2}, 3={}
        self.transitions = {
            ('q0', '0'): {'q0'},
            ('q0', '1'): {'q0'},
            ('q0', '0'): {'q1'},
            ('q1', '1'): {'q2'},
            ('q2', '0'): {'q0'},
            ('q2', '1'): {'q0'},
        }
    
    def accepts(self, string):
        current = {self.start_state}
        for char in string:
            next_states = set()
            for state in current:
                for inp, targets in self.transitions.items():
                    if inp == char and state in targets:
                        next_states.update(targets)
            current = next_states
        return bool(any(s in {'q2'} for s in current))

nfa = NFAEndsIn01()
test_nfa = [\"01\", \"001\", \"101\", \"010\", \"1001\", \"1101\"]
for s in test_nfa:
    print(f\"  '{s}' : {'ACCEPT' if nfa.accepts(s) else 'REJECT'}\")


## 3. Turing Machines

A **Turing machine** is a more powerful computational model with: a tape infinite in both directions, a head that reads/writes one symbol at a time, and a state transition function. Turing machines recognize recursively enumerable languages — the most general class.

### 3.1 Unary Decrementer: Turing Machine that Subtracts 1 from a Unary Number

Given a unary number n (represented by n ones on the tape), this Turing machine computes n-1 (decrements by 1).


In [ ]:
# Turing Machine: Unary Decrementer
# Tape format: _ 1 1 ... 1 _  (ones surrounded by blanks)
# States: q0 (start), q1 (decrementing), q2 (halt)
# Tape symbols: 1 (data), _ (blank/empty)

class TuringUnaryDecrement:
    \"\"\"Turing machine that decrements a unary number by 1.
    
    Input:  n ones on tape
    Output: n-1 ones on tape (if n > 0), or blank (if n = 0)
    \"\"\"

    def __init__(self):
        # Tape: infinite in both directions, indexed by int
        self.tape = {}  # dict mapping position -> symbol (default _ blank)
        self.head = 0  # head position
        self.state = 'q0'  # current state
        self.write_once = False  # only write blank once at end

    def write(self, symbol):
        \"\"\"Write a symbol at head position.\"\"\"
        self.tape[self.head] = symbol

    def move_left(self):
        \"\"\"Move head one position left.\"\"\"
        self.head -= 1

    def move_right(self):
        \"\"\"Move head one position right.\"\"\"
        self.head += 1

    def read(self):
        \"\"\"Read symbol at head position.\"\"\"
        return self.tape.get(self.head, '_')

    def transition(self, symbol):
        \"\"\"Execute one transition step based on current state and read symbol.\"\"\"
        if self.state == 'q0':
            if symbol == '1':
                self.write('_')  # erase last 1 (this is the decrement)
                self.move_left()
                self.state = 'q1'
            elif symbol == '_':
                # Tape was empty (input was 0 ones)
                self.write('1')  # write a single 1 (0-1 wraps to 1? actually should stay _)
                self.move_right()
                self.state = 'q2'
        elif self.state == 'q1':
            if symbol == '1':
                # Keep moving left over 1s
                self.move_left()
            elif symbol == '_':
                # Reached blank left of original tape
                self.move_right()  # back to the rightmost 1 position
                self.state = 'q2'
        elif self.state == 'q2':
            # Halt state - do nothing
            pass

    def run(self, num_ones=5):
        \"\"\"Run the decrementer with num_ones ones on the tape.\"\"\"
        # Initialize tape with num_ones ones starting at position 0
        for i in range(num_ones):
            self.write('1')

        max_steps = num_ones + 5  # safety limit
        for step in range(max_steps):
            symbol = self.read()
            self.transition(symbol)
            if self.state == 'q2':
                print(f\"  Halted after {step+1} steps\")
                return True
        print(f\"  Did not halt after {max_steps} steps\")
        return False

# Test: decrement 5 ones -> should give 4 ones
tm = TuringUnaryDecrement()
print(\"Turing Machine Unary Decrementer\")
print(\"=\" * 50)
print(\"Input: 5 ones (5)\")
tm.run(5)
print(\"Output tape:\")
for pos in range(-1, 6):
    symbol = tm.tape.get(pos, '_')
    print(f\"  pos {pos}: '{symbol}'\", end=\"\")
print()

# Verify: count remaining 1s
remaining = sum(1 for pos in tm.tape if tm.tape[pos] == '1')
print(f\"Remaining 1s: {remaining} (expected: 4)\")
"

## 4. Complexity Classes: P vs NP

### 4.1 Class P (Polynomial Time)

Problems solvable in polynomial time by a deterministic Turing machine:
- Sorting, finding shortest path, linear regression, pattern matching

### 4.2 Class NP (Nondeterministic Polynomial Time)

Problems where a proposed solution can be verified in polynomial time:
- Sudoku puzzle solution, Hamiltonian circuit, boolean satisfiability (SAT)

### 4.3 The Million-Dollar Question: P = NP?

Open problem: Is every problem whose solution can be quickly verified also quickly solvable?

### 4.4 NP-Complete Problems

If any NP-complete problem is in P, then P = NP. Key NP-complete problems:
- Boolean SAT (Satisfiability)
- Traveling Salesman Problem
- Knapsack Problem
- Graph Coloring


In [ ]:
# SAT solver: Check if a boolean formula in CNF is satisfiable
def solve_sat_simple(clauses, variables):
    \"\"\"Simple brute-force SAT solver.



## 5. Context-Free Grammars (CFG)

A CFG defines recursive languages (context-free). Production rules: NonTerminal -> Sequence of Terminals/NonTerminals.

### Example: Balanced Parentheses

S -> SS | (S) | ε

This grammar generates all properly balanced parenthesis strings.

### Parse Tree Example: (()())\"\

In [ ]:
# CYK algorithm: Parse string against CFG in Chomsky Normal Form
# CNF format: A -> BC | a (non-terminal -> two non-terminals, or non-terminal -> terminal)

def cyk_parse(cnf_rules, word):
    \"\"\"CYK (Cocke-Younger-Kasami) parsing algorithm.

    CNF rules: dict mapping non-terminal -> list of (A, B) pairs for A -> BC,
              plus set of (A, a) pairs for A -> a (terminals)
    word: input string as list of terminals

    n = len(word)
    # Parse[i][j] = set of non-terminals that generate word[i:i+j+1]
    Parse = [[set() for _ in range(n)] for _ in range(n)]

# Initialize: length 1 substrings
for i in range(n):
    for A, a in cnf_rules.get('terminals', []):
        if a == word[i]:
            Parse[i][i].add(A)

# Fill for increasing lengths
for length in range(1, n):  # length of substring minus 1
    for i in range(n - length):
        j = i + length
        for k in range(i, j):
            for A, B in cnf_rules.get('non_terminals', []):
                if B in Parse[i][k] and Parse[k+1][j]:
                    Parse[i][j].add(A)

# The start symbol (assumed 'S') accepts if it's in Parse[0][n-1]
return 'S' in Parse[0][n-1]

# Example CFG in CNF:
# S -> AB | BC
# A -> a
# B -> b
# C -> c
# This generates strings like \"ab\", \"bc\", \"abc\" (actually this specific
# grammar generates {ab, bc} but let's use a balanced parens example)

# Balanced parentheses CFG in CNF:
# S -> AB | a
# A -> SS | a
# B -> SS | b
# (generates: a, aa, ab, ba, bb, aaa, etc - simplified)

cnf_rules = {
    'terminals': {('S', 'a')},
    'non_terminals': [
        ('S', 'A'), ('S', 'B'),  # S -> AB
        ('A', 'S'), ('A', 'a'),  # A -> SS | a (simplified to just A -> a for CNF)
        ('B', 'S'), ('B', 'b'),  # B -> SS | b (simplified)
    ]
}

# Test: can 'a' be parsed?
print(f\"'a' parses: {cyk_parse(cnf_rules, ['a'])}\")

# Simpler: just S -> a terminal
cnf_simple = {
    'terminals': {('S', 'a')},
    'non_terminals': []
}
print(f\"'a' with S->a: {cyk_parse(cnf_simple, ['a'])}\")


## 6. Exercises

### Exercise 1: DFA Design
Design a DFA that accepts binary strings where the number of 0s is divisible by 3.

### Exercise 2: NFA to DFA Conversion
Convert the NFA that accepts strings ending in '101' to an equivalent DFA.

### Exercise 3: Turing Machine: Unary Incrementer
Design a Turing machine that increments a unary number by 1 (n ones -> n+1 ones).

### Exercise 4: NP-Completeness
Show that if you have a polynomial-time algorithm for the Hamiltonian Cycle problem, 
then you also have one for the Traveling Salesman Problem.

### Exercise 5: CYK Parsing
Given the CFG S -> SS | (S) | ε, use the CYK algorithm to determine if the string \"(())\" is generated.
